# WhatsApp to Excel Automation - Jupyter Notebook

This notebook automatically monitors your WhatsApp group and updates your Excel file with complaint data.

## Instructions:
1. Run each cell in order (Shift + Enter)
2. Keep this notebook open while it's running
3. Press the Stop button (⏹) to stop monitoring

---

## Step 1: Install Required Packages

Run this cell first to install all required packages.

In [9]:
# Install required packages
import subprocess
import sys

packages = ['selenium', 'openpyxl', 'webdriver-manager', 'pandas']

for package in packages:
    try:
        __import__(package.replace('-', '_'))
        print(f"✓ {package} already installed")
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
        print(f"✓ {package} installed")

print("\n✓ All packages installed!")

✓ selenium already installed
✓ openpyxl already installed
✓ webdriver-manager already installed
✓ pandas already installed

✓ All packages installed!


## Step 2: Import Libraries

In [10]:
import os
import time
import json
import re
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
import openpyxl
import pandas as pd

print("✓ All libraries imported successfully!")

✓ All libraries imported successfully!


## Step 3: Define the Automation Class

In [11]:
class WhatsAppExcelAutomation:
    def __init__(self, excel_file_path: str, group_name: str):
        """
        Initialize the automation system
        
        Args:
            excel_file_path: Path to your Excel file
            group_name: Name of WhatsApp group to monitor
        """
        self.excel_file_path = excel_file_path
        self.group_name = group_name
        self.driver = None
        self.processed_messages = set()
        self.load_processed_messages()
        
    def load_processed_messages(self):
        """Load already processed message IDs to avoid duplicates"""
        cache_file = Path("processed_messages.json")
        if cache_file.exists():
            with open(cache_file, 'r') as f:
                self.processed_messages = set(json.load(f))
    
    def save_processed_messages(self):
        """Save processed message IDs"""
        with open("processed_messages.json", 'w') as f:
            json.dump(list(self.processed_messages), f)
    
    def setup_driver(self):
        """Setup Selenium WebDriver for WhatsApp Web"""
        print("Setting up WhatsApp Web connection...")
        
        chrome_options = Options()
        chrome_options.add_argument("--no-sandbox")
        chrome_options.add_argument("--disable-dev-shm-usage")
        chrome_options.add_argument("--disable-gpu")
        chrome_options.add_argument("--start-maximized")
        
        # Create user data directory for persistent login
        user_data_dir = str(Path.home() / ".whatsapp_automation_jupyter")
        chrome_options.add_argument(f"user-data-dir={user_data_dir}")
        
        service = Service(ChromeDriverManager().install())
        self.driver = webdriver.Chrome(service=service, options=chrome_options)
        
        print("Opening WhatsApp Web...")
        self.driver.get("https://web.whatsapp.com")
        
        print("\n" + "="*60)
        print("IMPORTANT: Scan the QR code with your phone!")
        print("1. Open WhatsApp on your phone")
        print("2. Go to Settings > Linked Devices")
        print("3. Click 'Link a device'")
        print("4. Scan the QR code shown in the browser")
        print("="*60 + "\n")
        
        # Wait for user to scan QR code
        try:
            WebDriverWait(self.driver, 120).until(
                EC.presence_of_all_elements_located((By.XPATH, "//div[@data-testid='chat-list-item']"))
            )
            print("✓ Successfully logged in to WhatsApp Web!")
            time.sleep(3)
        except Exception as e:
            print(f"✗ Login failed or timed out: {e}")
            return False
        
        return True
    
    def find_group(self) -> bool:
        """Find and open the target WhatsApp group"""
        print(f"Looking for group: {self.group_name}...")
        
        try:
            # Click on search box
            search_box = WebDriverWait(self.driver, 10).until(
                EC.presence_of_element_located((By.XPATH, "//input[@placeholder='Search or start new chat']"))
            )
            search_box.click()
            search_box.clear()
            search_box.send_keys(self.group_name)
            
            time.sleep(2)
            
            # Click on the group
            group_item = WebDriverWait(self.driver, 10).until(
                EC.presence_of_element_located((By.XPATH, f"//span[contains(text(), '{self.group_name}')]"))
            )
            group_item.click()
            
            print(f"✓ Opened group: {self.group_name}")
            time.sleep(2)
            return True
            
        except Exception as e:
            print(f"✗ Could not find group: {e}")
            return False
    
    def extract_messages(self) -> List[Dict]:
        """Extract complaint messages from the group"""
        messages = []
        
        try:
            # Get all messages
            message_elements = self.driver.find_elements(By.XPATH, "//div[@data-testid='msg-container']")
            
            for msg_elem in message_elements:
                try:
                    # Extract message text
                    msg_text = msg_elem.text
                    
                    if not msg_text or len(msg_text) < 5:
                        continue
                    
                    # Create unique ID for message
                    msg_id = hash(msg_text) % ((2**31) - 1)
                    
                    if msg_id in self.processed_messages:
                        continue
                    
                    messages.append({
                        'id': msg_id,
                        'text': msg_text,
                        'timestamp': datetime.now().isoformat()
                    })
                    
                    self.processed_messages.add(msg_id)
                    
                except Exception as e:
                    continue
            
            return messages
            
        except Exception as e:
            print(f"Error extracting messages: {e}")
            return []
    
    def parse_complaint_data(self, message_text: str) -> Optional[Dict]:
        """Parse complaint data from message text"""
        
        lines = message_text.strip().split('\n')
        if len(lines) < 2:
            return None
        
        data = {}
        
        # Extract time
        time_match = re.search(r'(\d{1,2}):(\d{2})\s*(am|pm|AM|PM)?', lines[0])
        if time_match:
            data['Time'] = time_match.group(0)
        
        # Extract date
        date_match = re.search(r'(\d{1,2})-(\d{1,2})-(\d{4})', message_text)
        if date_match:
            data['Date'] = date_match.group(0)
        
        # Extract branch
        branch_match = re.search(r'([A-Za-z\s]+)\s+Branch', message_text, re.IGNORECASE)
        if branch_match:
            data['Branch'] = branch_match.group(1).strip()
        
        # Extract complaint
        complaint_text = '\n'.join(lines[1:])
        if complaint_text:
            data['Complain'] = complaint_text.strip()[:200]
        
        # Categorize
        complaint_lower = complaint_text.lower()
        if 'staff' in complaint_lower or 'employee' in complaint_lower:
            data['Category'] = 'staff issue'
        elif 'food' in complaint_lower or 'quality' in complaint_lower:
            data['Category'] = 'quality issue'
        elif 'service' in complaint_lower or 'slow' in complaint_lower:
            data['Category'] = 'service issue'
        elif 'waste' in complaint_lower:
            data['Category'] = 'wastage issue'
        else:
            data['Category'] = 'other'
        
        data['Response (Y / N)'] = 'N'
        
        return data if len(data) > 2 else None
    
    def update_excel(self, complaint_data: Dict):
        """Add complaint data to Excel file"""
        
        try:
            # Load existing workbook
            if not os.path.exists(self.excel_file_path):
                print(f"Excel file not found: {self.excel_file_path}")
                return False
            
            wb = openpyxl.load_workbook(self.excel_file_path)
            ws = wb.active
            
            # Find the next empty row
            next_row = ws.max_row + 1
            
            # Define column mapping
            columns = {
                'Date': 1,
                'Time': 2,
                'Branch': 3,
                'Category': 4,
                'Classification': 5,
                'Stations ': 6,
                'Complain': 7,
                'Response (Y / N)': 8,
                "Manager,s Reply": 9,
                'Shift Manager': 10,
                'Employe Name': 11,
                'Employe ID': 12,
                'Fine': 13,
            }
            
            # Write data to Excel
            for key, col_num in columns.items():
                if key in complaint_data:
                    ws.cell(row=next_row, column=col_num, value=complaint_data[key])
            
            # Save workbook
            wb.save(self.excel_file_path)
            print(f"✓ Added complaint to Excel (Row {next_row})")
            return True
            
        except Exception as e:
            print(f"✗ Error updating Excel: {e}")
            return False
    
    def run_continuous_monitoring(self, check_interval: int = 30):
        """
        Run continuous monitoring of WhatsApp group
        
        Args:
            check_interval: Seconds between each check (default: 30 seconds)
        """
        
        if not self.setup_driver():
            return
        
        if not self.find_group():
            self.driver.quit()
            return
        
        print(f"\n✓ Starting continuous monitoring (checking every {check_interval} seconds)...")
        print("To stop monitoring, interrupt the kernel (Kernel > Interrupt)\n")
        
        try:
            iteration = 0
            while True:
                iteration += 1
                print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] Check #{iteration}")
                
                try:
                    # Extract new messages
                    messages = self.extract_messages()
                    
                    if messages:
                        print(f"  Found {len(messages)} new message(s)")
                        
                        for msg in messages:
                            # Parse complaint data
                            complaint_data = self.parse_complaint_data(msg['text'])
                            
                            if complaint_data:
                                print(f"  Processing: {complaint_data.get('Complain', 'Unknown')[:40]}...")
                                
                                # Update Excel
                                self.update_excel(complaint_data)
                    else:
                        print("  No new messages")
                    
                    # Save processed messages
                    self.save_processed_messages()
                    
                    # Wait before next check
                    time.sleep(check_interval)
                    
                except Exception as e:
                    print(f"  Error during monitoring: {e}")
                    time.sleep(check_interval)
        
        except KeyboardInterrupt:
            print("\n\nStopping monitoring...")
        
        finally:
            self.driver.quit()
            print("✓ Closed WhatsApp Web connection")

print("✓ Automation class defined!")

✓ Automation class defined!


## Step 4: Configure Your Settings

**Edit these values with your information:**

In [20]:
# ===== CONFIGURATION =====
# Change these values to match your setup

# Path to your Excel file
# EXCEL_FILE_PATH = "C:\\Users\\GT-Tech\\Desktop\\Feb_surveillance.xlsx" # Windows example
# # EXCEL_FILE_PATH = "/Users/YourName/Documents/Feb_surveillance.xlsm"  # Mac example
# # EXCEL_FILE_PATH = "/home/username/Documents/Feb_surveillance.xlsm"  # Linux example

# # WhatsApp group name to monitor
# GROUP_NAME = "Surveillance HNS"  # Change this to your group name

# # Check interval in seconds (how often to check for new messages)
# CHECK_INTERVAL = 30  # Check every 30 seconds

# ===== END CONFIGURATION =====
EXCEL_FILE_PATH =r"C:\Users\GT-Tech\Desktop\Feb_surveillance.xlsm.xlsx"
GROUP_NAME = "Hns Surveillance"
CHECK_INTERVAL = 30


print(f"Configuration:")
print(f"  Excel File: {EXCEL_FILE_PATH}")
print(f"  Group Name: {GROUP_NAME}")
print(f"  Check Interval: {CHECK_INTERVAL} seconds")
print(f"\n✓ Configuration ready!")

Configuration:
  Excel File: C:\Users\GT-Tech\Desktop\Feb_surveillance.xlsm.xlsx
  Group Name: Hns Surveillance
  Check Interval: 30 seconds

✓ Configuration ready!


## Step 5: Start Monitoring

**Run this cell to start the automation!**

**Important:**
- Keep this notebook open
- Don't close the browser window
- To stop: Click the Stop button (⏹) or go to Kernel > Interrupt

In [22]:
# Verify Excel file exists
if not os.path.exists(EXCEL_FILE_PATH):
    print(f"✗ ERROR: Excel file not found at: {EXCEL_FILE_PATH}")
    print("Please update EXCEL_FILE_PATH in the configuration cell above")
else:
    print(f"✓ Excel file found: {EXCEL_FILE_PATH}")
    print(f"✓ Group name: {GROUP_NAME}")
    print(f"\n" + "="*60)
    print("STARTING AUTOMATION...")
    print("="*60 + "\n")
    
    # Create and run automation
    automation = WhatsAppExcelAutomation(EXCEL_FILE_PATH, GROUP_NAME)
    automation.run_continuous_monitoring(CHECK_INTERVAL)

✓ Excel file found: C:\Users\GT-Tech\Desktop\Feb_surveillance.xlsm.xlsx
✓ Group name: Hns Surveillance

STARTING AUTOMATION...

Setting up WhatsApp Web connection...
Opening WhatsApp Web...

IMPORTANT: Scan the QR code with your phone!
1. Open WhatsApp on your phone
2. Go to Settings > Linked Devices
3. Click 'Link a device'
4. Scan the QR code shown in the browser

✗ Login failed or timed out: Message: 



## Step 6: Stop Monitoring

**To stop the automation:**
1. Click the **Stop button (⏹)** in the toolbar above
2. Or go to **Kernel > Interrupt Kernel**

The script will close the browser and stop monitoring.

In [18]:
import os
from pathlib import Path

# Desktop کا path
desktop_path = Path.home() / "Desktop"
print(desktop_path)

# تمام Excel فائلیں دیکھیں
for file in desktop_path.glob("*.xlsx"):
    print(file)


C:\Users\GT-Tech\Desktop
C:\Users\GT-Tech\Desktop\Feb_surveillance.xlsm.xlsx


---

## Troubleshooting

### Issue: "Excel file not found"
- Make sure you updated the `EXCEL_FILE_PATH` in Step 4
- Use the full path to your file

### Issue: "Chrome not found"
- Install Google Chrome from: https://www.google.com/chrome/

### Issue: "Cannot find group"
- Make sure the group name in Step 4 matches exactly
- Check spelling and spaces

### Issue: "QR code not loading"
- Check your internet connection
- Try running the cell again

### Issue: "Notebook keeps disconnecting"
- Don't close the browser window
- Keep your computer awake (disable sleep mode)
- Don't close the notebook tab

---

## Tips

✓ Keep the notebook open while monitoring  
✓ Don't close the browser window  
✓ Keep your computer awake  
✓ Close your Excel file (or it might not update)  
✓ Check interval of 30 seconds is good for most cases  

---

**Enjoy automated complaint tracking!** 🎉